<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-02-grafos/11%20-%20Modelagem%20de%20Esteiras%20e%20Triagem%20como%20Grafo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 11 - Notebook: Modelagem Topológica de Transporte e Triagem como Dígrafos Ponderados
## Célula de Manufatura Flexível (FMS)

Neste notebook implementamos a classe base **`GrafoManufatura`** para representar a malha de esteiras transportadoras e os postos de triagem pneumática como um **Grafo Dirigido e Ponderado (Dígrafo)** $G=(V, E, W)$.

In [1]:
from typing import List, Dict, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII pura alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[float]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII com representação clara de adjacências."""
    larguras = [max(len(str(r)), 8) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else f"{v:.1f}"
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoManufatura:
    """
    Representa a malha física de esteiras e postos de triagem da FMS como Dígrafo Ponderado G = (V, E, W).
    """
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)

        # Matriz binária de adjacência (1 se há trajeto direto u -> v, 0 caso contrário)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]

        # Matriz de adjacência ponderada por distâncias físicas (comprimento em metros)
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0

        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_rota(self, origem: str, destino: str, comprimento_m: float,
                        tag_atuador: str, mecanismo: str = "Esteira"):
        """Adiciona um segmento de transporte direcionado e ponderado entre dois postos."""
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]

        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m

        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador ISA": tag_atuador,
            "Mecanismo": mecanismo
        })

    def obter_graus(self) -> List[Dict[str, Any]]:
        """Calcula o grau de entrada (deg-) e grau de saída (deg+) de cada posto de trabalho."""
        graus = []
        for i, v in enumerate(self.vertices):
            deg_out = sum(self.adj_binaria[i])
            deg_in = sum(self.adj_binaria[r][i] for r in range(self.n))
            graus.append({
                "Posto de Trabalho": v,
                "Grau Entrada (deg-)": deg_in,
                "Grau Saída (deg+)": deg_out
            })
        return graus


# --- 1. DEFINIÇÃO TOPOLÓGICA DOS POSTOS DA FMS ---
postos_fms = [
    "SILO-100",   # Silo de alimentação de peças brutas
    "INSP-200",   # Túnel com sensores de geometria (ZS-201/202) e cor (AS-201..203)
    "DIV-201",    # Ponto de desvio 1 (Pistão XV-201)
    "DIV-202",    # Ponto de desvio 2 (Pistão XV-202)
    "DIV-203",    # Ponto de desvio 3 (Pistão XV-203)
    "CX-301",     # Caixa 1 (Grandes Vermelhas - US-301)
    "CX-302",     # Caixa 2 (Grandes Verdes - US-302)
    "CX-303",     # Caixa 3 (Pequenas Azuis - US-303)
    "DESC-304"    # Calha terminal de descarte (Peças não conformes)
]

fms_rede = GrafoManufatura(postos_fms)

# --- 2. CADASTRO DAS ROTAS COM TAGS DE ATUADORES E COMPRIMENTOS ---
fms_rede.adicionar_rota("SILO-100", "INSP-200", 0.8, "XV-101", "Alimentador Pneumático")
fms_rede.adicionar_rota("INSP-200", "DIV-201",  1.2, "M-101",  "Esteira Principal")
fms_rede.adicionar_rota("DIV-201",  "CX-301",   0.5, "XV-201", "Pistão de Triagem 1")
fms_rede.adicionar_rota("DIV-201",  "DIV-202",  1.0, "M-101",  "Esteira Principal")
fms_rede.adicionar_rota("DIV-202",  "CX-302",   0.5, "XV-202", "Pistão de Triagem 2")
fms_rede.adicionar_rota("DIV-202",  "DIV-203",  1.0, "M-101",  "Esteira Principal")
fms_rede.adicionar_rota("DIV-203",  "CX-303",   0.5, "XV-203", "Pistão de Triagem 3")
fms_rede.adicionar_rota("DIV-203",  "DESC-304", 0.8, "M-101",  "Calha de Descarte")

# --- 3. RELATÓRIOS FORMATADOS EM ASCII ---
print("=== TABELA DE ROTAS DE TRANSPORTE E TRIAGEM (FMS) ===")
print(formatar_tabela(fms_rede.arestas_detalhes))

print("\n=== GRAUS TOPOLÓGICOS DOS POSTOS DE TRABALHO ===")
print(formatar_tabela(fms_rede.obter_graus()))

print("\n=== MATRIZ DE ADJACÊNCIA PONDERADA (COMPRIMENTO EM METROS) ===")
print(formatar_matriz(fms_rede.adj_pesos, fms_rede.vertices, fms_rede.vertices))

# --- 4. ASSERÇÕES DETERMINÍSTICAS DE INTEGRIDADE ---
assert len(fms_rede.vertices) == 9, "Erro: Quantidade de postos diverge do projeto."
assert len(fms_rede.arestas_detalhes) == 8, "Erro: Quantidade de trechos físicos de esteiras diverge."
assert fms_rede.adj_pesos[fms_rede.v_to_idx["DIV-201"]][fms_rede.v_to_idx["CX-301"]] == 0.5
assert fms_rede.adj_pesos[fms_rede.v_to_idx["SILO-100"]][fms_rede.v_to_idx["INSP-200"]] == 0.8
assert fms_rede.adj_binaria[fms_rede.v_to_idx["DIV-202"]][fms_rede.v_to_idx["CX-302"]] == 1
assert fms_rede.adj_binaria[fms_rede.v_to_idx["CX-301"]][fms_rede.v_to_idx["DIV-201"]] == 0, "Erro: Esteiras são estritamente unidirecionais."

print("\n[OK] Grafo da Célula de Manufatura Flexível estruturado e validado com sucesso!")

=== TABELA DE ROTAS DE TRANSPORTE E TRIAGEM (FMS) ===
Origem   | Destino  | Comprimento (m) | Atuador ISA | Mecanismo             
---------+----------+-----------------+-------------+-----------------------
SILO-100 | INSP-200 | 0.8             | XV-101      | Alimentador Pneumático
INSP-200 | DIV-201  | 1.2             | M-101       | Esteira Principal     
DIV-201  | CX-301   | 0.5             | XV-201      | Pistão de Triagem 1   
DIV-201  | DIV-202  | 1.0             | M-101       | Esteira Principal     
DIV-202  | CX-302   | 0.5             | XV-202      | Pistão de Triagem 2   
DIV-202  | DIV-203  | 1.0             | M-101       | Esteira Principal     
DIV-203  | CX-303   | 0.5             | XV-203      | Pistão de Triagem 3   
DIV-203  | DESC-304 | 0.8             | M-101       | Calha de Descarte     

=== GRAUS TOPOLÓGICOS DOS POSTOS DE TRABALHO ===
Posto de Trabalho | Grau Entrada (deg-) | Grau Saída (deg+)
------------------+---------------------+------------------
SILO-1